### Task 0: Set up Your Environment and Load API Keys

In [1]:
from dotenv import load_dotenv 
import os  
load_dotenv()

True

In [2]:
from typing import List, Sequence

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import requests
import xml.etree.ElementTree as ET

In [3]:
import time

def arxiv_search_tool(query: str, max_results: int = 3) -> str:
    """
    Search papers from arXiv related to the given query.
    Returns a human-readable summary of results.
    """
    base_url = "https://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results
    }

    # Send a request to the arXiv API, retrying on timeouts and non-200 responses.
    # arXiv is slow and rate-limited (it asks for ~3s between requests), so back off between attempts.
    response, last_error = None, None
    for attempt in range(3):
        try:
            response = requests.get(base_url, params=params, timeout=30)
            if response.status_code == 200:
                break
            last_error = f"status {response.status_code}"
        except requests.RequestException as e:
            response, last_error = None, str(e)
        if attempt < 2:
            time.sleep(3 * (attempt + 1))

    if response is None or response.status_code != 200:
        return f"Error: Unable to fetch data from arXiv ({last_error})"

    # Parse the XML response
    root = ET.fromstring(response.text)
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    entries = root.findall("atom:entry", ns)

    if not entries:
        return f"No papers found for topic '{query}'."

    output = [f"Papers for '{query}':"]
    for i, entry in enumerate(entries, start=1):
        title = entry.find("atom:title", ns).text.strip()
        summary = entry.find("atom:summary", ns).text.strip().replace("\n", " ")
        link = entry.find("atom:id", ns).text.strip()
        output.append(f"{i}. {title}\n   Link: {link}\n   Abstract: {summary[:300]}...")

    return "\n".join(output)

# Test the function
print(arxiv_search_tool("AI in dental"))

Papers for 'AI in dental':
1. Faith in AI can narrow the futures individuals consider
   Link: http://arxiv.org/abs/2603.28944v2
   Abstract: Artificial intelligence (AI) predictions are increasingly used to inform human decisions. Here, using a behavioral implementation of the classic Newcomb's paradox in 1,305 participants, we show that AI predictions can also shape the reasoning people use to make a decision. In this paradigm, perceive...
2. Foundations of GenIR
   Link: http://arxiv.org/abs/2501.02842v1
   Abstract: The chapter discusses the foundational impact of modern generative AI models on information access (IA) systems. In contrast to traditional AI, the large-scale training and superior data modeling of generative AI models enable them to produce high-quality, human-like responses, which brings brand ne...
3. Competing Visions of Ethical AI: A Case Study of OpenAI
   Link: http://arxiv.org/abs/2601.16513v1
   Abstract: Introduction. AI Ethics is framed distinctly across act

In [ ]:
# Create the model client shared by all agents (Gemini via its OpenAI-compatible API; reads GEMINI_API_KEY from .env).
# AutoGen 0.7.5 doesn't know gemini-3.8-flash yet, so the endpoint and model capabilities must be given explicitly.
import json
import httpx
from autogen_core.models import ModelFamily, ModelInfo


class GeminiThoughtSignatureTransport(httpx.AsyncHTTPTransport):
    """Gemini 3 rejects tool-call history without a thought_signature, which AutoGen doesn't send back.
    Attach Google's documented placeholder signature to every past tool call before the request goes out."""

    async def handle_async_request(self, request):
        if request.method == "POST" and request.url.path.endswith("/chat/completions"):
            body = json.loads(request.content)
            for msg in body.get("messages", []):
                for call in msg.get("tool_calls") or []:
                    call.setdefault("extra_content", {"google": {"thought_signature": "skip_thought_signature_validator"}})
            headers = {k: v for k, v in request.headers.items() if k.lower() != "content-length"}
            request = httpx.Request(request.method, request.url, headers=headers, json=body, extensions=request.extensions)
        return await super().handle_async_request(request)


model_client = OpenAIChatCompletionClient(
    model="gemini-3.8-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    http_client=httpx.AsyncClient(transport=GeminiThoughtSignatureTransport(), timeout=60),
    model_info=ModelInfo(
        vision=True,
        function_calling=True,
        json_output=True,
        structured_output=True,
        family=ModelFamily.UNKNOWN,
        multiple_system_messages=False,
    ),
)

In [8]:
# Create the topic refinement agent
topic_agent = AssistantAgent(
    "TopicAgent",
    description="Refines a broad research idea into a specific topic.",
    model_client=model_client,
    system_message="""
    ROLE: You are a Research Topic Refiner.
    TASK: Given a general idea, output one refined research topic.

    OUTPUT FORMAT:
    Refined Research Topic: <Topic>
    End with: TASK_COMPLETED_TOPIC_AGENT
    """
)

topic_result = await topic_agent.run(task="AI in healthcare")
print(topic_result.messages[-1].content)

Refined Research Topic: Federated Learning Frameworks for Privacy-Preserving Diagnostic Imaging and Clinical Decision Support

TASK_COMPLETED_TOPIC_AGENT
